# Production Agent Architecture

**Level:** Advanced · **Time:** 60 min

Building an agent in a notebook is easy. Deploying it to a Kubernetes cluster where pods crash, networks drop, and queues back up is hard.

In this notebook, we will simulate four critical production patterns:
1. **Durable Execution (Checkpointing):** An agent saves its state to a database before "crashing", and successfully resumes on reboot.
2. **Idempotency Enforcement:** An agent hallucinates and tries to charge a credit card twice for the same transaction. The Tool Gateway prevents the double-charge.
3. **Dead Letter Queues (DLQs):** A toxic message crashes the worker repeatedly until it is routed to a DLQ for human inspection.
4. **Concurrency & Rate Limiting:** Protecting an external API from an autoscaled swarm of agent workers.

---
## Pattern 1: Durable Execution (Checkpointers)

An agent proposes a Terraform deployment and needs human approval. Instead of blocking the thread (`time.sleep`), it saves its state to a simulated PostgreSQL database and exits. When the "human" approves, a new worker resumes the exact state.

In [1]:
# Simulated PostgreSQL Checkpointer
db_checkpoints = {}

def execute_agent_step(session_id, current_state):
    print(f"\n[Worker Node 1] Starting execution for Session {session_id}...")
    
    if current_state == "INIT":
        print("[Worker Node 1] Step 1: Drafted Terraform Plan.")
        print("[Worker Node 1] Requires HITL Approval. Saving Checkpoint to DB and exiting thread.")
        # Checkpointing!
        db_checkpoints[session_id] = "AWAITING_APPROVAL"
        return
        
    if current_state == "APPROVED":
        print(f"[Worker Node 2] Step 2: Resuming from checkpoint.")
        print(f"[Worker Node 2] Executing Terraform Apply... Success!")
        db_checkpoints[session_id] = "COMPLETED"
        return

# 1. Start the job
execute_agent_step("session-123", "INIT")

# 2. Server dies, time passes. Human clicks "Approve" in the UI.
print("\n--- 3 Days Later: Human clicks 'Approve' ---")
db_checkpoints["session-123"] = "APPROVED"

# 3. A totally different worker node picks up the webhook event
execute_agent_step("session-123", db_checkpoints["session-123"])



[Worker Node 1] Starting execution for Session session-123...
[Worker Node 1] Step 1: Drafted Terraform Plan.
[Worker Node 1] Requires HITL Approval. Saving Checkpoint to DB and exiting thread.

--- 3 Days Later: Human clicks 'Approve' ---

[Worker Node 1] Starting execution for Session session-123...
[Worker Node 2] Step 2: Resuming from checkpoint.
[Worker Node 2] Executing Terraform Apply... Success!


---
## Pattern 2: Idempotency Enforcement

An LLM receives a network timeout while calling `charge_credit_card`. Assuming the call failed, the LLM retries. Because the orchestrator attached a strict `ToolCallID` (Idempotency Key), the Tool Gateway catches the duplicate and prevents a double charge.

In [2]:
# Simulated Redis Cache for Idempotency
idempotency_cache = {}

def charge_credit_card(amount: int, tool_call_id: str) -> str:
    print(f"\n[Tool Gateway] Intercepted tool call ID: {tool_call_id}")
    
    # Check Idempotency Cache
    if tool_call_id in idempotency_cache:
        print("[Tool Gateway] CACHE HIT! Duplicate UUID detected. Returning cached response to prevent double-charge.")
        return idempotency_cache[tool_call_id]
        
    # Execute actual business logic
    print(f"[Banking API] Charging card for ${amount}...")
    success_response = "200 OK: Card Charged Successfully."
    
    # Save to cache
    idempotency_cache[tool_call_id] = success_response
    return success_response


# Attempt 1: The agent calls the tool. It succeeds, but the network drops before the agent sees the response.
print("[Agent] Calling charge_credit_card...")
charge_credit_card(amount=50, tool_call_id="call_abc123")
print("[Agent] Network Timeout! I didn't get a response. I will retry.")

# Attempt 2: The agent retries with the SAME tool call ID.
print("\n[Agent] RETRYING charge_credit_card...")
result = charge_credit_card(amount=50, tool_call_id="call_abc123")
print(f"[Agent] Received Response: {result}")


[Agent] Calling charge_credit_card...

[Tool Gateway] Intercepted tool call ID: call_abc123
[Banking API] Charging card for $50...
[Agent] Network Timeout! I didn't get a response. I will retry.

[Agent] RETRYING charge_credit_card...

[Tool Gateway] Intercepted tool call ID: call_abc123
[Tool Gateway] CACHE HIT! Duplicate UUID detected. Returning cached response to prevent double-charge.
[Agent] Received Response: 200 OK: Card Charged Successfully.


---
## Pattern 3: Dead Letter Queues (DLQ)

If a user submits a payload so toxic or malformed that it crashes the worker node, the queue will retry it. If it crashes 3 times in a row, the message is routed to a Dead Letter Queue (DLQ) for human inspection, rather than looping infinitely and consuming all cluster compute.

In [3]:
import time

message_queue = [{"payload": "valid_job"}, {"payload": "TOXIC_MALFORMED_JOB"}]
dead_letter_queue = []

def process_queue():
    while message_queue:
        job = message_queue.pop(0)
        retries = 0
        max_retries = 3
        
        while retries < max_retries:
            print(f"\n[Queue Worker] Processing job: {job['payload']} (Attempt {retries + 1}/{max_retries})")
            
            try:
                if job['payload'] == "TOXIC_MALFORMED_JOB":
                    raise Exception("FATAL: Out of Memory / Null Pointer")
                print(f"[Queue Worker] SUCCESS: Job {job['payload']} completed.")
                break # Exit retry loop on success
            except Exception as e:
                print(f"[Queue Worker] CRASH: {e}")
                retries += 1
                
        if retries == max_retries:
            print(f"\n[Message Queue] Job failed {max_retries} times. Routing to DLQ.")
            dead_letter_queue.append(job)

process_queue()
print(f"\n[System] Current DLQ Contents: {dead_letter_queue}")



[Queue Worker] Processing job: valid_job (Attempt 1/3)
[Queue Worker] SUCCESS: Job valid_job completed.

[Queue Worker] Processing job: TOXIC_MALFORMED_JOB (Attempt 1/3)
[Queue Worker] CRASH: FATAL: Out of Memory / Null Pointer

[Queue Worker] Processing job: TOXIC_MALFORMED_JOB (Attempt 2/3)
[Queue Worker] CRASH: FATAL: Out of Memory / Null Pointer

[Queue Worker] Processing job: TOXIC_MALFORMED_JOB (Attempt 3/3)
[Queue Worker] CRASH: FATAL: Out of Memory / Null Pointer

[Message Queue] Job failed 3 times. Routing to DLQ.

[System] Current DLQ Contents: [{'payload': 'TOXIC_MALFORMED_JOB'}]


---
## Pattern 4: Concurrency & Rate Limiting

If your agent workers auto-scale to 100 pods, they might accidentally launch a Denial of Service (DoS) attack on a weak internal API. The Tool Gateway must enforce rate limiting and backpressure.

In [4]:
import time

# Simulated API Gateway bucket (Token Bucket algorithm)
tokens_available = 2

def call_internal_api(agent_id: str):
    global tokens_available
    print(f"[API Gateway] Agent {agent_id} requesting API access...")
    
    if tokens_available > 0:
        tokens_available -= 1
        print(f"[API Gateway] ✅ 200 OK: Request granted. (Tokens left: {tokens_available})")
        return True
    else:
        print(f"[API Gateway] ❌ 429 TOO MANY REQUESTS: Rate limit exceeded. Back off!")
        return False

# An autoscaled swarm of 5 agents all try to hit the API at the exact same millisecond
print("--- Autoscaled Swarm Hits the API ---")
for i in range(1, 6):
    call_internal_api(f"Worker-{i}")


--- Autoscaled Swarm Hits the API ---
[API Gateway] Agent Worker-1 requesting API access...
[API Gateway] ✅ 200 OK: Request granted. (Tokens left: 1)
[API Gateway] Agent Worker-2 requesting API access...
[API Gateway] ✅ 200 OK: Request granted. (Tokens left: 0)
[API Gateway] Agent Worker-3 requesting API access...
[API Gateway] ❌ 429 TOO MANY REQUESTS: Rate limit exceeded. Back off!
[API Gateway] Agent Worker-4 requesting API access...
[API Gateway] ❌ 429 TOO MANY REQUESTS: Rate limit exceeded. Back off!
[API Gateway] Agent Worker-5 requesting API access...
[API Gateway] ❌ 429 TOO MANY REQUESTS: Rate limit exceeded. Back off!
